<a id="setup"></a>
# <p style="background-color: #ff6200; font-family:calibri; color:white; font-size:140%; font-family:Verdana; text-align:center; border-radius:15px 50px;">Chapter 3 | Filter and Join Queries</p>

In [2]:
import importlib
from collection_lab import Collection
from database import Database
from queryOperator import QueryOperator

<a id="libraries"></a>
# <b><span style='color:#fcc36d'>0|</span><span style='color:#ff6200'> Mock Databases construction </span></b>

##### Sub-bloc of the Database

In [3]:
# Sub blocs Category, Supplier, etc...
# Category
schema_category = {
    "type": "object",
    "properties": {"title": {"type": "string"}},
    "required": ["title"]
}

# Supplier
schema_supplier = {
    "type": "object",
    "properties": {
        "IDS": {"type": "integer"},
        "name": {"type": "string"},
        "SIRET": {"type": "string"},
        "headOffice": {"type": "string"},
        "Revenue": {"type": "integer"}
    },
    "required": ["IDS", "name", "SIRET", "headOffice", "Revenue"]
}

# Price
schema_price = {
    "type": "object",
    "properties": {
        "amount": {"type": "number"},
        "currency": {"type": "string"},
        "VAT": {"type": "number"}
    },
    "required": ["amount", "currency", "VAT"]
}

# Product (Categories + Suppliers embedded)
product_embedded_schema = {
    "type": "object",
    "properties": {
        "IDP": {"type": "integer"},
        "name": {"type": "string"},
        "brand": {"type": "string"},
        "description": {"type": "string"},
        "image_url": {"type": "string"},
        "price": schema_price,
        # Nesting [Cat]
        "categories": {"type": "array", "items": schema_category}, 
        # Nesting Supp
        "supplier": schema_supplier                               
    },
    "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
}

# Warehouse 
warehouse_schema = {
    "type": "object", 
    "properties": {
        "IDW": {"type": "integer"}, 
        "address": {"type": "string"},
        "capacity": {"type": "integer"} 
    },
    "required": ["IDW", "address", "capacity"]
}

# Stock
stock_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"}, 
        "IDW": {"type": "integer"}, 
        "quantity": {"type": "integer"},
        "location": {"type": "string"}
    },
    "required": ["IDP", "IDW", "quantity", "location"]    
}

# Order Line
order_line_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"},
        "IDC": {"type": "integer"},
        "date": {"type": "string"},
        "quantity": {"type": "integer"},
        "deliveryDate": {"type": "string"},
        "comment": {"type": "string"},
        "grade": {"type": "integer"}
    },
    "required": ["IDP", "IDC", "quantity", "date", "deliveryDate", "comment", "grade"]
}

# Client
client_schema = {
    "type": "object", 
    "properties": {
        "IDC": {"type": "integer"},
        "ln": {"type": "string"},
        "fn": {"type": "string"},   
        "address": {"type": "string"},
        "nationality": {"type": "string"},
        "birthDate": {"type": "string"},
        "email": {"type": "string"}
    },
    "required": ["IDC", "ln", "fn", "address", "nationality", "birthDate", "email"]
}

#### Mock DBs Schema

In [4]:
# Stats 

STATS = {
    "nb_products": 10**5,        # 100,000 Products
    "nb_clients": 10**7,         # 10 Million Clients
    "nb_warehouses": 200,        # 200 Warehouses
    "nb_orderlines": 4 * 10**9,  # 4 Billion Order Lines
    "nb_brands": 5000,
    # Derived stats for array multipliers
    "avg_cat_per_prod": 2        # Average categories per product
}

In [5]:
# Mock DB1

db1_schemas = {
    "Product": {
        "type": "object",
        "properties": {
            "IDP": {"type": "integer"},
            "name": {"type": "string"},
            "brand": {"type": "string"},
            "description": {"type": "string"}, 
            "image_url": {"type": "string"},   
            "price": schema_price,
            
            # Nesting: Array of Categories
            "categories": {"type": "array", "items": schema_category},
            
            # Nesting: Supplier Object
            "supplier": schema_supplier 
        },
        "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
    },
    
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "OrderLine": order_line_schema,
    "Client": client_schema
}

db1 = Database("DB1 - Normalized")

# Create Collections
c_prod = Collection("Product", db1_schemas["Product"], stats=STATS)
c_stock = Collection("Stock", db1_schemas["Stock"], stats=STATS, count_rule="product_x_warehouse")
c_warehouse = Collection("Warehouse", db1_schemas["Warehouse"], stats=STATS)
c_orderline = Collection("OrderLine", db1_schemas["OrderLine"], stats=STATS)
c_client = Collection("Client", db1_schemas["Client"], stats=STATS)

# Add to DB
db1.add_collection(c_prod)
db1.add_collection(c_stock)
db1.add_collection(c_warehouse)
db1.add_collection(c_orderline)
db1.add_collection(c_client)

In [6]:
# Mock DB2

db4_schemas = {
    "OrderLine": {
        "type": "object",
        "properties": {
            "IDC": {"type": "integer"},
            "date": {"type": "string"},
            "quantity": {"type": "integer"},
            "deliveryDate": {"type": "string"},
            "comment": {"type": "string"},
            "grade": {"type": "integer"},
            
            # Nesting: Product + Categories + Supplier
            "product": product_embedded_schema
        },
        "required": ["IDC", "date", "quantity", "grade", "product"]
    },
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "Client": client_schema
}

db4 = Database("DB4 - Denormalized")

# Create Collections
c_orderline_embedded = Collection("OrderLine", db4_schemas["OrderLine"], stats=STATS)
c_stock_db4 = Collection("Stock", db4_schemas["Stock"], stats=STATS, count_rule="product_x_warehouse")
c_warehouse_db4 = Collection("Warehouse", db4_schemas["Warehouse"], stats=STATS)
c_client_db4 = Collection("Client", db4_schemas["Client"], stats=STATS)

# Add to DB
db4.add_collection(c_orderline_embedded)
db4.add_collection(c_stock_db4)
db4.add_collection(c_warehouse_db4)
db4.add_collection(c_client_db4)


<a id="libraries"></a>
# <b><span style='color:#fcc36d'>1 | </span><span style='color:#ff6200'> Validation test for the lab 2 </span></b>

In [7]:
print("=== SCENARIO 1: FILTER ON DB1 (LIGHTWEIGHT PRODUCTS) ===")
# Case: Searching for all products of brand "Apple" (5% of the catalog)
# Issue: Data is sharded by "IDP" (Product ID), not by brand.
# Expected Result: Broadcast (all servers scanned), but documents are small.

res_filter_db1 = QueryOperator.op_filter(
    collection=c_prod,               # DB1 Collection (Small size per doc)
    filter_key="brand",              # Filtering on Brand
    selectivity=0.05,                # 5% of products kept
    sharding_key="IDP",              # Sharded by ID (so Brand != ID -> Broadcast)
    projected_keys=["name", "price"] # We only want name and price
)

print(f"Filter DB1 (Brand='Apple'):")
print(f" > Scanned: {res_filter_db1['scanned_gb']:.4f} GB (Disk Input)")
print(f" > Output:  {res_filter_db1['output_gb']:.4f} GB (Network Output)")
print(f" > Servers: {res_filter_db1['servers_involved']} (Broadcast)")
print("-" * 50)


print("\n=== SCENARIO 2: FILTER ON DB4 (ORDERLINES WITH EMBEDDED PRODUCTS) ===")
# Case: Searching for orders on a specific date.
# Context: In DB4, "OrderLine" contains the WHOLE product (huge documents).
# Expected Result: Scanned volume (Input) should be MUCH larger than DB1.

res_filter_db4 = QueryOperator.op_filter(
    collection=c_orderline_embedded, # DB4 Collection (Huge documents)
    filter_key="date",
    selectivity=0.01,                # 1% of orders
    sharding_key="IDC"               # Sharded by Client or Order ID
)

print(f"Filter DB4 (Date='2023-01-01'):")
print(f" > Scanned: {res_filter_db4['scanned_gb']:.4f} GB (Watch out for the volume!)")
print(f" > Output:  {res_filter_db4['output_gb']:.4f} GB")
print("-" * 50)


print("\n=== SCENARIO 3: THE JOIN NIGHTMARE (DB1) ===")
# Case: To display orders in DB1, we must join OrderLine + Product.
# This is what DB4 avoids thanks to embedding.
# Expected Result: Astronomical numbers in GB scanned (The cost of Normalization).

res_join_db1 = QueryOperator.op_nested_loop(
    outer_col=c_orderline,       # Outer Table (4 Billion rows)
    inner_col=c_prod,            # Inner Table (100k products)
    join_key="IDP",              # Join Key
    sharding_key_inner="IDP"     # Product is sharded by IDP (Targeted Lookup)
)

print(f"JOIN DB1 (OrderLine + Product):")
print(f" > Total Work: {res_join_db1['total_scanned_gb']:.2f} GB")
print(f" > Comment: {res_join_db1['comment']}")

=== SCENARIO 1: FILTER ON DB1 (LIGHTWEIGHT PRODUCTS) ===
Filter DB1 (Brand='Apple'):
 > Scanned: 0.1330 GB (Disk Input)
 > Output:  0.0011 GB (Network Output)
 > Servers: 1000 (Broadcast)
--------------------------------------------------

=== SCENARIO 2: FILTER ON DB4 (ORDERLINES WITH EMBEDDED PRODUCTS) ===
Filter DB4 (Date='2023-01-01'):
 > Scanned: 6616.1156 GB (Watch out for the volume!)
 > Output:  66.1612 GB
--------------------------------------------------

=== SCENARIO 3: THE JOIN NIGHTMARE (DB1) ===
JOIN DB1 (OrderLine + Product):
 > Total Work: 533297.66 GB
 > Comment: Optimized Join (Key = Shard Key)
